# Deal data: a quick look with SQL

A short exploration of the raw CSVs in `data/` using DuckDB SQL. It reads the **raw CSVs** (not `data.duckdb`)
so it shows the data before cleaning. The cleaning itself lives in `src/clean_data.py`.

In [ ]:
import duckdb
from pathlib import Path

# Works whether the notebook is started from the project root or from notebooks/
DATA = Path("data") if Path("data").exists() else Path("../data")

con = duckdb.connect()  # in-memory, so this never locks data.duckdb
for name in ["accounts", "deals", "events"]:
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('{(DATA / f'{name}.csv').as_posix()}')")

sql = lambda q: con.sql(q).df()

## 1. Size of each table

In [ ]:
sql('''
    SELECT 'accounts' AS table_name, count(*) AS rows FROM accounts
    UNION ALL SELECT 'deals',  count(*) FROM deals
    UNION ALL SELECT 'events', count(*) FROM events
''')

## 2. Deal outcomes
The model predicts `status`. Note how few `open` deals there are compared to won/lost.

In [ ]:
sql('''
    SELECT status,
           count(*)                                         AS deals,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           round(median(amount))                            AS median_amount
    FROM deals
    GROUP BY status
    ORDER BY deals DESC
''')

## 3. Win rate by lead source
Win rate uses closed deals only (won / (won + lost)).

In [ ]:
sql('''
    SELECT lead_source,
           count(*)                                              AS closed_deals,
           round(100.0 * avg(CASE WHEN status = 'won' THEN 1 ELSE 0 END), 1) AS win_rate_pct
    FROM deals
    WHERE status IN ('won', 'lost')
    GROUP BY lead_source
    ORDER BY win_rate_pct DESC
''')

## 4. Win rate by industry and funding stage

In [ ]:
sql('''
    SELECT a.industry,
           count(*)                                                          AS closed_deals,
           round(100.0 * avg(CASE WHEN d.status = 'won' THEN 1 ELSE 0 END), 1) AS win_rate_pct
    FROM deals d
    JOIN accounts a USING (account_id)
    WHERE d.status IN ('won', 'lost')
    GROUP BY a.industry
    ORDER BY win_rate_pct DESC
''')

In [ ]:
sql('''
    SELECT a.funding_stage,
           count(*)                                                          AS closed_deals,
           round(100.0 * avg(CASE WHEN d.status = 'won' THEN 1 ELSE 0 END), 1) AS win_rate_pct
    FROM deals d
    JOIN accounts a USING (account_id)
    WHERE d.status IN ('won', 'lost')
    GROUP BY a.funding_stage
    ORDER BY win_rate_pct DESC
''')

## 5. Engagement before the deal was created
Events in the 90 days **before** `created_date`. Only events before the deal starts are counted,
so this could be used as a model feature without leaking the outcome.

In [ ]:
sql('''
    WITH engagement AS (
        SELECT d.deal_id, d.status,
               count(e.event_type)                                        AS events_90d,
               count(e.event_type) FILTER (WHERE e.event_type = 'demo_request') AS demos_90d
        FROM deals d
        LEFT JOIN events e
               ON e.account_id = d.account_id
              AND e.event_date >= d.created_date - INTERVAL 90 DAY
              AND e.event_date <  d.created_date
        GROUP BY d.deal_id, d.status
    )
    SELECT status,
           round(avg(events_90d), 2) AS avg_events_90d,
           round(avg(demos_90d), 2)  AS avg_demos_90d,
           round(100.0 * avg(CASE WHEN demos_90d > 0 THEN 1 ELSE 0 END), 1) AS pct_with_demo
    FROM engagement
    GROUP BY status
    ORDER BY status
''')

## 6. Data quality checks
These are what `src/clean_data.py` fixes (duplicates) or keeps on purpose (missing optional values).

In [ ]:
sql('''
    SELECT 'events: exact duplicate rows' AS check_name,
           (SELECT count(*) FROM events) - (SELECT count(*) FROM (SELECT DISTINCT * FROM events)) AS count
    UNION ALL SELECT 'accounts: duplicate account_id', count(*) - count(DISTINCT account_id) FROM accounts
    UNION ALL SELECT 'deals: duplicate deal_id',       count(*) - count(DISTINCT deal_id)    FROM deals
    UNION ALL SELECT 'deals: account not in accounts', count(*) FROM deals WHERE account_id NOT IN (SELECT account_id FROM accounts)
    UNION ALL SELECT 'deals: close_date before created_date', count(*) FROM deals WHERE close_date < created_date
    UNION ALL SELECT 'deals: open with no close_date (expected)', count(*) FROM deals WHERE status = 'open' AND close_date IS NULL
    UNION ALL SELECT 'accounts: missing annual_revenue_musd', count(*) FROM accounts WHERE annual_revenue_musd IS NULL
    UNION ALL SELECT 'accounts: missing growth_rate_yoy',     count(*) FROM accounts WHERE growth_rate_yoy IS NULL
    UNION ALL SELECT 'accounts: missing hiring_velocity',     count(*) FROM accounts WHERE hiring_velocity IS NULL
''')

## Takeaways
- **Industry and funding stage matter a lot**: software wins ~55% of closed deals vs ~26% for manufacturing; series B ~50% vs bootstrapped ~21%.
- **Lead source barely matters** (all channels ~37-39% win rate).
- **Engagement is a strong signal**: won deals had ~6.7 events in the 90 days before the deal vs ~3.9 for lost ones.
  The current model doesn't use `events` at all, so this is a likely feature to add.
- `open` deals are only ~3% of the data, so the model will see very few `open` examples.
- The only real dirt is **591 duplicate event rows**; `src/clean_data.py` removes them.
- Missing revenue / growth / hiring values are left as NULL: the form treats them as optional and the tree model handles missing values.

Next step: run `python src/clean_data.py` from the project root to write the cleaned tables into `data/data.duckdb`.


In [ ]:
con.close()